# Task 1: News Topic Classifier Using BERT

## Objective
The objective of this project is to build a News Topic Classification system using the **BERT (Bidirectional Encoder Representations from Transformers)** model. The AG News dataset is used to fine-tune a pre-trained BERT model for classifying news articles into different categories. The project covers data preprocessing, tokenization, model training, evaluation using Accuracy and F1-Score, and deployment through a Gradio web application.

## Problem Statement
Automatically classifying news articles into predefined categories is an important Natural Language Processing (NLP) task. This project aims to leverage the power of transformer-based models, specifically BERT, to accurately classify news headlines into categories such as World, Sports, Business, and Science/Technology.

## Project Workflow
1. Install and import required libraries.
2. Load the AG News dataset.
3. Perform data preprocessing and tokenization.
4. Fine-tune the BERT model.
5. Evaluate the model using Accuracy and F1-Score.
6. Make predictions on unseen news articles.
7. Deploy the model using Gradio.

In [14]:
!pip uninstall -y torchvision torchaudio -q

In [15]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


In [24]:
import datasets.config as ds_config
ds_config.TORCHVISION_AVAILABLE = False

In [16]:
MODEL_NAME = "bert-base-uncased"
OUTPUT_DIR = "./bert-ag-news"
MAX_LENGTH = 128
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42

# Small subset -> runs in a few minutes, good for a first error-free pass.
# Set both to None once you want the full 120k/7.6k run.
TRAIN_SUBSET = 2000
EVAL_SUBSET = 500

LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
label2id = {name: i for i, name in enumerate(LABEL_NAMES)}

In [17]:
raw_datasets = load_dataset("fancyzhx/ag_news")

if TRAIN_SUBSET:
    raw_datasets["train"] = raw_datasets["train"].shuffle(seed=SEED).select(range(TRAIN_SUBSET))
if EVAL_SUBSET:
    raw_datasets["test"] = raw_datasets["test"].shuffle(seed=SEED).select(range(EVAL_SUBSET))

print(raw_datasets)
print(raw_datasets["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 500
    })
})
{'text': 'Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.', 'label': 0}


In [18]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_datasets = raw_datasets.map(tokenize_fn, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_datasets)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 500
    })
})


In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

print(model.config.num_labels, "labels:", model.config.id2label)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


4 labels: {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}


In [20]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    precision, recall, _, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "precision_macro": precision,
        "recall_macro": recall,
    }

In [21]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,   # renamed from `tokenizer=` in newer transformers
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro
1,0.607062,0.394242,0.884000,0.885460,0.884154,0.890683,0.884569
2,0.285732,0.394535,0.882000,0.882931,0.881790,0.885208,0.883273
3,0.179222,0.391240,0.888000,0.888816,0.887785,0.891729,0.888384


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=375, training_loss=0.4352432657877604, metrics={'train_runtime': 142.2957, 'train_samples_per_second': 42.166, 'train_steps_per_second': 2.635, 'total_flos': 282537013614336.0, 'train_loss': 0.4352432657877604, 'epoch': 3.0})

In [26]:
metrics = trainer.evaluate()
for k, v in metrics.items():
    print(f"{k}: {v}")

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro
0.179222,0.391240,3,0.888000,0.888816,0.887785,0.891729,0.888384


eval_loss: 0.39123955368995667
eval_accuracy: 0.888
eval_f1_macro: 0.8888160030232712
eval_f1_weighted: 0.8877853079810268
eval_precision_macro: 0.8917292490118577
eval_recall_macro: 0.8883841330537395


In [27]:
predictions = trainer.predict(tokenized_datasets["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))
print("Confusion matrix (rows=true, cols=predicted):")
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

       World     0.9333    0.8167    0.8711       120
      Sports     0.9600    0.9917    0.9756       121
    Business     0.8478    0.8731    0.8603       134
    Sci/Tech     0.8258    0.8720    0.8482       125

    accuracy                         0.8880       500
   macro avg     0.8917    0.8884    0.8888       500
weighted avg     0.8900    0.8880    0.8878       500

Confusion matrix (rows=true, cols=predicted):
[[ 98   4  11   7]
 [  1 120   0   0]
 [  0   1 117  16]
 [  6   0  10 109]]


In [28]:
def predict_topic(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
    return {name: float(p) for name, p in zip(LABEL_NAMES, probs)}

sample_headlines = [
    "Manchester United wins dramatic final in extra time",
    "Federal Reserve raises interest rates amid inflation concerns",
    "NASA announces new mission to explore Europa",
]

for headline in sample_headlines:
    print(headline, "->", predict_topic(headline))

Manchester United wins dramatic final in extra time -> {'World': 0.019233334809541702, 'Sports': 0.9698902368545532, 'Business': 0.006403485778719187, 'Sci/Tech': 0.004472875501960516}
Federal Reserve raises interest rates amid inflation concerns -> {'World': 0.02629401907324791, 'Sports': 0.00652445200830698, 'Business': 0.9564516544342041, 'Sci/Tech': 0.010729923844337463}
NASA announces new mission to explore Europa -> {'World': 0.01595010608434677, 'Sports': 0.010700601153075695, 'Business': 0.012805238366127014, 'Sci/Tech': 0.9605440497398376}


In [29]:
import gradio as gr

def classify(headline):
    if not headline or not headline.strip():
        return {name: 0.0 for name in LABEL_NAMES}
    return predict_topic(headline)

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=3, placeholder="Enter a news headline...", label="Headline / Text"),
    outputs=gr.Label(num_top_classes=4, label="Predicted Topic"),
    title="AG News Topic Classifier (Fine-tuned BERT)",
    description="Fine-tuned bert-base-uncased on AG News. Classes: World, Sports, Business, Sci/Tech.",
    examples=[
        "Manchester United wins dramatic final in extra time",
        "Federal Reserve raises interest rates amid inflation concerns",
        "NASA announces new mission to explore Europa",
        "Peace talks resume between the two nations after ceasefire",
    ],
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://000fe80f6cbd7ff55a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
